# 00 — First principles of sales outreach

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~35 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/02_sales_intelligence/00_first_principles.ipynb)

**Prerequisites**:
- Basic Python and statistics knowledge
- Familiarity with NLP concepts (embeddings, similarity)

**What you will learn**:
- Why personalization depth drives response rates in outbound sales
- How to quantify outreach quality with a scoring model
- The economics of SDR time and where AI creates leverage
- How to build A/B testing frameworks for email experiments

In [ ]:
# @title Setup — Run this cell first
# --- Google Colab Setup ---
!pip install -q "sentence-transformers>=2.7.0" "numpy>=1.24.0" "scipy>=1.11.0"

import numpy as np
from dataclasses import dataclass, field
from typing import Optional
import random
import math

random.seed(42)
np.random.seed(42)
print("Setup complete.")

## Section 1 — What makes sales outreach effective?

Effective outreach sits at the intersection of three forces:

```
Outreach Impact = Relevance × Timing × Value Proposition
```

- **Relevance**: Does the message address the prospect's actual situation?
- **Timing**: Does it arrive when the prospect is actively feeling the pain?
- **Value proposition**: Does it clearly articulate what changes for them?

Most outreach fails because it optimizes only one axis — usually value proposition — while ignoring relevance entirely. Let's see the difference concretely.

In [ ]:
# Three outreach examples targeting the same prospect
prospect = {
    "name": "Sarah Chen",
    "title": "VP of Engineering",
    "company": "DataFlow Inc",
    "company_size": 200,
    "recent_news": "Just raised Series B, hiring 15 engineers",
    "tech_stack": ["Python", "Kubernetes", "PostgreSQL"],
    "pain_signals": ["slow deployment cycles", "scaling infrastructure"],
}

generic_email = """Hi Sarah,

I wanted to reach out because our platform helps companies like yours
improve their engineering workflow. Would you be open to a quick chat?

Best, Alex"""

personalized_email = """Hi Sarah,

Congrats on DataFlow's Series B! With 15 new engineers joining,
onboarding and deployment velocity often become bottlenecks.

Our platform helps engineering teams cut deployment cycles by 40%.
Would 15 minutes next week work to explore if that's relevant for DataFlow?

Best, Alex"""

deep_research_email = """Hi Sarah,

I noticed DataFlow just closed your Series B — congrats! I saw you're
hiring 15 engineers, and given your Kubernetes + PostgreSQL stack,
I'd guess scaling the CI/CD pipeline for that many new contributors
is top of mind.

We helped TechCorp (similar stack, 180 engineers) reduce their
deployment cycle from 45 min to 12 min after a similar growth phase.

Would it be useful to see how they structured their pipeline? Happy
to share the case study — no pitch, just the architecture.

Best, Alex"""

print("=== GENERIC ===")
print(generic_email)
print("\n=== PERSONALIZED ===")
print(personalized_email)
print("\n=== DEEPLY RESEARCHED ===")
print(deep_research_email)

## Section 2 — The personalization spectrum

Personalization exists on a spectrum with measurable depth levels:

| Level | Description | Example | Typical response rate |
|-------|-------------|---------|----------------------|
| 0 | No personalization | "Dear Sir/Madam" | <1% |
| 1 | Surface | Name + company | 1–2% |
| 2 | Contextual | Recent news, hiring, funding | 5–8% |
| 3 | Deep | Inferred pain points from signals | 12–20%+ |

Let's build a scorer that classifies outreach into these levels.

In [ ]:
from sentence_transformers import SentenceTransformer
from numpy.linalg import norm

model = SentenceTransformer("all-MiniLM-L6-v2")

# Personalization signal categories
SURFACE_SIGNALS = ["first name", "company name", "job title"]
CONTEXTUAL_SIGNALS = ["recent funding", "hiring plans", "company news",
                      "industry trend", "product launch"]
DEEP_SIGNALS = ["inferred pain point", "tech stack challenge",
                "competitor comparison", "specific metric", "case study match"]

def score_personalization_depth(email_text: str, prospect: dict) -> dict:
    """Score an email's personalization depth on a 0-3 scale."""
    score = 0.0
    signals_found = []

    # Surface checks
    if prospect["name"].split()[0].lower() in email_text.lower():
        score += 0.5
        signals_found.append("first name")
    if prospect["company"].lower() in email_text.lower():
        score += 0.5
        signals_found.append("company name")

    # Contextual checks via keyword overlap
    contextual_keywords = ["series b", "raised", "hiring", "engineers",
                           "funding", "growth"]
    ctx_hits = sum(1 for kw in contextual_keywords if kw in email_text.lower())
    if ctx_hits >= 2:
        score += 1.0
        signals_found.append(f"contextual ({ctx_hits} keywords)")

    # Deep checks — specific technical or pain references
    deep_keywords = ["kubernetes", "postgresql", "deployment cycle",
                     "ci/cd", "pipeline", "case study", "architecture"]
    deep_hits = sum(1 for kw in deep_keywords if kw in email_text.lower())
    if deep_hits >= 2:
        score += 1.0
        signals_found.append(f"deep ({deep_hits} keywords)")

    level = min(int(score), 3)
    return {"score": round(score, 2), "level": level, "signals": signals_found}

# Score all three emails
for label, email in [("Generic", generic_email),
                     ("Personalized", personalized_email),
                     ("Deep Research", deep_research_email)]:
    result = score_personalization_depth(email, prospect)
    print(f"{label:15s} → Level {result['level']} (score={result['score']}) "
          f"signals={result['signals']}")

## Section 3 — Why mail merge fails

Mail merge is the dominant approach to outbound: a template with `{first_name}` and `{company}` placeholders. It scales but produces Level-0 to Level-1 personalization. Let's build one, see the output, and then calculate why the economics are brutal.

In [ ]:
# Simple mail merge engine
TEMPLATES = [
    """Hi {first_name},

I'm reaching out because {company} might benefit from our platform.
Companies in the {industry} space often struggle with {generic_pain}.
Would you be open to a quick 15-minute call?

Best, Alex""",
    """Hi {first_name},

I noticed {company} is growing — exciting times! Our solution helps
{industry} teams move faster. Would love to show you how.

Cheers, Alex""",
]

prospects = [
    {"first_name": "Sarah", "company": "DataFlow Inc",
     "industry": "data infrastructure", "generic_pain": "scaling"},
    {"first_name": "Marcus", "company": "HealthBridge",
     "industry": "healthtech", "generic_pain": "compliance"},
    {"first_name": "Priya", "company": "RetailEdge",
     "industry": "retail tech", "generic_pain": "integration"},
]

print("=== MAIL MERGE OUTPUT ===\n")
for i, p in enumerate(prospects):
    template = TEMPLATES[i % len(TEMPLATES)]
    email = template.format(**p)
    print(f"--- To: {p['first_name']} at {p['company']} ---")
    print(email)
    print()

In [ ]:
# The economics of generic outreach
@dataclass
class OutreachEconomics:
    emails_per_day: int = 100
    response_rate: float = 0.02  # 2%
    meeting_rate: float = 0.30   # 30% of responses become meetings
    close_rate: float = 0.20     # 20% of meetings close
    deal_value: float = 50_000   # average deal
    sdr_annual_cost: float = 85_000  # salary + benefits + tools
    working_days: int = 250

    def annual_revenue(self) -> float:
        """Calculate expected annual revenue from this outreach model."""
        daily_responses = self.emails_per_day * self.response_rate
        daily_meetings = daily_responses * self.meeting_rate
        daily_deals = daily_meetings * self.close_rate
        return daily_deals * self.deal_value * self.working_days

    def roi(self) -> float:
        return self.annual_revenue() / self.sdr_annual_cost

generic = OutreachEconomics(response_rate=0.02)
personalized = OutreachEconomics(emails_per_day=40, response_rate=0.12)

print(f"{'Metric':<30s} {'Generic':>12s} {'Personalized':>14s}")
print("-" * 58)
print(f"{'Emails/day':<30s} {generic.emails_per_day:>12d} {personalized.emails_per_day:>14d}")
print(f"{'Response rate':<30s} {generic.response_rate:>11.0%} {personalized.response_rate:>13.0%}")
print(f"{'Annual revenue':<30s} ${generic.annual_revenue():>10,.0f}  ${personalized.annual_revenue():>12,.0f}")
print(f"{'ROI (revenue/cost)':<30s} {generic.roi():>11.1f}x {personalized.roi():>13.1f}x")
print(f"\nPersonalized outreach: fewer emails, dramatically better economics.")

## Section 4 — Lead qualification as multi-signal scoring

Not all prospects are equal. Before personalizing outreach, we must score leads to focus effort. A lead score combines three signal families:

| Signal family | Examples | Weight rationale |
|---------------|----------|------------------|
| **Intent** | Job postings, tech changes, funding | Strongest predictor — they're actively moving |
| **Fit** | Industry, company size, tech stack | Baseline qualification |
| **Engagement** | Website visits, content downloads | Shows existing awareness |

In [ ]:
@dataclass
class LeadSignals:
    """Raw signals for a prospect, scored on 0-1 scale."""
    # Intent signals
    recent_funding: float = 0.0
    hiring_relevant_roles: float = 0.0
    tech_stack_change: float = 0.0
    # Fit signals
    industry_match: float = 0.0
    company_size_fit: float = 0.0
    tech_stack_overlap: float = 0.0
    # Engagement signals
    website_visits: float = 0.0
    content_downloads: float = 0.0

WEIGHTS = {
    "intent": {"recent_funding": 0.20, "hiring_relevant_roles": 0.15,
               "tech_stack_change": 0.15},
    "fit": {"industry_match": 0.15, "company_size_fit": 0.10,
            "tech_stack_overlap": 0.10},
    "engagement": {"website_visits": 0.08, "content_downloads": 0.07},
}

def score_lead(signals: LeadSignals) -> dict:
    """Compute a weighted lead score from raw signals."""
    intent = (signals.recent_funding * WEIGHTS["intent"]["recent_funding"]
              + signals.hiring_relevant_roles * WEIGHTS["intent"]["hiring_relevant_roles"]
              + signals.tech_stack_change * WEIGHTS["intent"]["tech_stack_change"])
    fit = (signals.industry_match * WEIGHTS["fit"]["industry_match"]
           + signals.company_size_fit * WEIGHTS["fit"]["company_size_fit"]
           + signals.tech_stack_overlap * WEIGHTS["fit"]["tech_stack_overlap"])
    engagement = (signals.website_visits * WEIGHTS["engagement"]["website_visits"]
                  + signals.content_downloads * WEIGHTS["engagement"]["content_downloads"])
    total = intent + fit + engagement
    return {"total": round(total, 3), "intent": round(intent, 3),
            "fit": round(fit, 3), "engagement": round(engagement, 3)}

# Sample prospects
sample_leads = {
    "DataFlow Inc": LeadSignals(
        recent_funding=0.9, hiring_relevant_roles=0.8, tech_stack_change=0.3,
        industry_match=0.9, company_size_fit=0.7, tech_stack_overlap=0.8,
        website_visits=0.5, content_downloads=0.2),
    "OldCorp LLC": LeadSignals(
        recent_funding=0.0, hiring_relevant_roles=0.1, tech_stack_change=0.0,
        industry_match=0.4, company_size_fit=0.3, tech_stack_overlap=0.2,
        website_visits=0.0, content_downloads=0.0),
    "ScaleUp AI": LeadSignals(
        recent_funding=0.7, hiring_relevant_roles=0.9, tech_stack_change=0.6,
        industry_match=0.8, company_size_fit=0.9, tech_stack_overlap=0.7,
        website_visits=0.8, content_downloads=0.6),
}

print(f"{'Company':<18s} {'Total':>7s} {'Intent':>8s} {'Fit':>6s} {'Engage':>8s}")
print("-" * 50)
for name, signals in sample_leads.items():
    s = score_lead(signals)
    print(f"{name:<18s} {s['total']:>7.3f} {s['intent']:>8.3f} {s['fit']:>6.3f} {s['engagement']:>8.3f}")

## Section 5 — The feedback loop

Every sent email generates data: was it opened, did the recipient reply, was a meeting booked? This creates a feedback loop. The key tool is A/B testing — splitting prospects into groups and measuring which approach performs better.

The critical statistical question: **When do we have enough data to trust the result?**

In [ ]:
from scipy import stats

def simulate_ab_test(
    n_a: int, rate_a: float, n_b: int, rate_b: float
) -> dict:
    """Simulate an A/B test and compute statistical significance."""
    successes_a = np.random.binomial(n_a, rate_a)
    successes_b = np.random.binomial(n_b, rate_b)
    p_a = successes_a / n_a
    p_b = successes_b / n_b

    # Two-proportion z-test
    p_pool = (successes_a + successes_b) / (n_a + n_b)
    se = math.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
    z = (p_b - p_a) / se if se > 0 else 0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))

    return {"p_a": round(p_a, 4), "p_b": round(p_b, 4),
            "lift": round((p_b - p_a) / p_a * 100, 1) if p_a > 0 else 0,
            "z": round(z, 3), "p_value": round(p_value, 4),
            "significant": p_value < 0.05}

def required_sample_size(
    baseline_rate: float, min_detectable_effect: float,
    alpha: float = 0.05, power: float = 0.80
) -> int:
    """Compute required sample size per group for a two-proportion test."""
    p1 = baseline_rate
    p2 = baseline_rate * (1 + min_detectable_effect)
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(power)
    p_avg = (p1 + p2) / 2
    n = ((z_alpha * math.sqrt(2 * p_avg * (1 - p_avg))
          + z_beta * math.sqrt(p1*(1-p1) + p2*(1-p2)))**2
         / (p2 - p1)**2)
    return math.ceil(n)

# Run a simulated test
result = simulate_ab_test(n_a=500, rate_a=0.02, n_b=500, rate_b=0.05)
print("A/B Test Simulation:")
print(f"  Group A: {result['p_a']:.2%}  |  Group B: {result['p_b']:.2%}")
print(f"  Lift: {result['lift']}%  |  p-value: {result['p_value']}")
print(f"  Significant at 0.05: {result['significant']}")

print(f"\nSample sizes needed to detect various lifts (baseline=2%):")
for lift_pct in [25, 50, 100, 200]:
    n = required_sample_size(0.02, lift_pct / 100)
    print(f"  {lift_pct:>3d}% lift → {n:>6,d} emails per group")

## Section 6 — Market analysis

The sales engagement market exceeds $5B, driven by clear unit economics. Let's break down the numbers that explain why AI outreach is compelling.

In [ ]:
# SDR time allocation analysis
time_allocation = {
    "Prospecting/research": 0.21,
    "Writing emails": 0.17,
    "Data entry / CRM": 0.14,
    "Internal meetings": 0.13,
    "Actual selling (calls/demos)": 0.35,
}

sdr_cost = 87_500  # salary + benefits + tools
hours_per_year = 2000

print("=== SDR Time Allocation ===")
print(f"{'Activity':<35s} {'% Time':>8s} {'Hours/yr':>10s} {'Cost':>10s}")
print("-" * 65)
for activity, pct in time_allocation.items():
    hours = hours_per_year * pct
    cost = sdr_cost * pct
    print(f"{activity:<35s} {pct:>7.0%} {hours:>10.0f} ${cost:>9,.0f}")

automatable = sum(v for k, v in time_allocation.items()
                  if k in ["Prospecting/research", "Writing emails", "Data entry / CRM"])
print(f"\nAutomatable by AI: {automatable:.0%} of time = "
      f"${sdr_cost * automatable:,.0f}/yr per SDR")

# AI outreach cost model
print("\n=== AI Outreach Cost Model ===")
api_cost_per_prospect = 0.08  # research + generation
prospects_per_month = 2000
monthly_api_cost = api_cost_per_prospect * prospects_per_month
annual_api_cost = monthly_api_cost * 12
print(f"API cost per researched prospect: ${api_cost_per_prospect:.2f}")
print(f"Prospects per month:              {prospects_per_month:,d}")
print(f"Annual API cost:                  ${annual_api_cost:,.0f}")
print(f"SDR cost for same work:           ${sdr_cost * automatable:,.0f}")
print(f"Savings:                          ${sdr_cost * automatable - annual_api_cost:,.0f}/yr")

## Exercises

1. **Improve the personalization scorer** — The current scorer uses simple keyword matching. Extend it to use embedding similarity: encode personalization signal categories and compare them against email content. Does the embedding-based scorer rank the three sample emails in the same order?

2. **Build a timing model** — Create a `TimingScore` class that estimates how receptive a prospect is *right now* based on signals like "days since funding announcement," "active job postings," and "recent tech blog post." Combine it with the lead score to create a prioritized outreach queue.

3. **Multi-armed bandit for email templates** — Instead of a simple A/B test, implement a Thompson Sampling bandit that allocates more sends to better-performing templates over time. Simulate 1,000 email sends across 4 templates and plot the cumulative regret.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Outreach effectiveness = Relevance × Timing × Value; most tools optimize only one. |
| 2 | Personalization has measurable depth levels: surface (1–2% reply), contextual (5–8%), deep (12–20%+). |
| 3 | Mail merge economics are poor: high volume, low conversion wastes SDR time at $85K+/yr. |
| 4 | Lead scoring combines intent, fit, and engagement signals with configurable weights. |
| 5 | A/B testing email outreach requires surprisingly large sample sizes — plan for 5,000+ per variant. |
| 6 | AI can automate 52% of SDR time (research, writing, data entry) at ~$0.08/prospect. |

## What's Next

- **Next**: [01_architecture.ipynb](./01_architecture.ipynb) — Design the research agent, scoring model, and email generation pipeline
- **Related**: [02_build.ipynb](./02_build.ipynb) — Implement the full sales intelligence agent

## References & Further Reading

1. [Salesforce, "State of Sales Report," 2024](https://www.salesforce.com/resources/research-reports/state-of-sales/) — SDR time allocation data and productivity benchmarks
2. [Gong Labs, "The Data Behind Cold Email Response Rates," 2023](https://www.gong.io/blog/cold-email-response-rates/) — Response rate benchmarks by personalization level
3. [HubSpot, "Sales Statistics," 2024](https://www.hubspot.com/sales-statistics) — Comprehensive sales outreach metrics and trends
4. [Evan Miller, "Sample Size Calculator"](https://www.evanmiller.org/ab-testing/sample-size.html) — A/B test power analysis reference